# 5. Backbone check

Repeats the condition comparison with EfficientNet-B0 instead of MobileNetV1, on the same folds,
masks and shuffles, without GA. The CNN head and the ensemble on all 512 head features are scored
on pooled out-of-fold predictions. MobileNetV1 features from notebook 1 are scored the same way.
Run after notebook 1, which writes `images.npz`, `masks.pkl` and the MobileNetV1 features.

In [ ]:
from pathlib import Path

import cv2
import joblib
import numpy as np
import pandas as pd
import pywt
import tensorflow as tf
from scipy.stats import binomtest
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import f1_score, recall_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

DATASET = "BUSI"  # or "BUS-UCLM"
OUT_DIR = Path("../outputs") / DATASET
BACKBONE = "EfficientNetB0"  # or "ResNet50"
CHECK_DIR = OUT_DIR / "backbone_check" / BACKBONE
CHECK_DIR.mkdir(parents=True, exist_ok=True)

CONDITIONS = ["raw", "predicted", "shuffled", "oracle"]
SIZE, SEED, N_AUG = 224, 42, 5
WEIGHTS = [1, 1, 2, 1]
BACKBONES = {
    "EfficientNetB0": (tf.keras.applications.EfficientNetB0, tf.keras.applications.efficientnet.preprocess_input),
    "ResNet50": (tf.keras.applications.ResNet50, tf.keras.applications.resnet50.preprocess_input),
}

data = joblib.load(OUT_DIR / "folds.pkl")
y, folds = data["y"], data["folds"]
images = np.load(OUT_DIR / "images.npz")
rgb, gray = images["rgb"], images["gray"]
masks = joblib.load(OUT_DIR / "masks.pkl")

## Inputs (same fusion and shuffling as notebook 1)

In [ ]:
def haar_fusion(image, mask):
    a1, d1 = pywt.dwt2(image.astype(float), "haar")
    a2, d2 = pywt.dwt2(mask.astype(float), "haar")
    details = tuple(np.where(np.abs(u) > np.abs(v), u, v) for u, v in zip(d1, d2))
    fused = np.clip(pywt.idwt2(((a1 + a2) / 2, details), "haar"), 0, 255).astype(np.uint8)
    return np.repeat(fused[..., None], 3, axis=2)


def shuffled_masks(k):
    rng = np.random.default_rng(SEED + k)
    out = masks["expert"].copy()
    for split in folds[k]:
        for c in range(3):
            idx = split[y[split] == c]
            out[idx] = masks["expert"][rng.permutation(idx)]
    return out


def condition_images(condition, k):
    if condition == "raw":
        return rgb
    m = {"oracle": lambda: masks["expert"],
         "predicted": lambda: masks["predicted"][k],
         "shuffled": lambda: shuffled_masks(k)}[condition]()
    return np.stack([haar_fusion(g, mm) for g, mm in zip(gray, m)])

## Backbone activations with a fixed augmentation pool

In [ ]:
def augment(img, rng):
    angle, zoom = rng.uniform(-20, 20), rng.uniform(0.9, 1.1)
    m = cv2.getRotationMatrix2D((SIZE / 2, SIZE / 2), angle, zoom)
    m[0, 1] += rng.uniform(-0.1, 0.1)
    m[:, 2] += rng.uniform(-0.15, 0.15, 2) * SIZE
    out = cv2.warpAffine(img, m, (SIZE, SIZE), borderMode=cv2.BORDER_REPLICATE)
    return out[:, ::-1].copy() if rng.random() < 0.5 else out


def activations(imgs_in, net, preprocess):
    views = []
    for v in range(1 + N_AUG):
        imgs = imgs_in if v == 0 else np.stack(
            [augment(im, np.random.default_rng([SEED, i, v])) for i, im in enumerate(imgs_in)])
        batches = [net(preprocess(imgs[s:s + 64].astype("float32")), training=False).numpy()
                   for s in range(0, len(imgs), 64)]
        views.append(np.concatenate(batches).astype("float16"))
    return np.stack(views)

## Dense-512 head and ensemble (as in notebooks 1 and 2)

In [ ]:
def train_head(acts, train, val, seed):
    layers, l2 = tf.keras.layers, tf.keras.regularizers.l2(0.01)
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    inp = layers.Input((acts.shape[2],))
    x = layers.Dense(512, "relu", kernel_regularizer=l2, name="f512")(inp)
    x = layers.BatchNormalization()(layers.Dropout(0.3)(x))
    x = layers.Dense(120, "relu", kernel_regularizer=l2)(x)
    x = layers.BatchNormalization()(layers.Dropout(0.2)(x))
    model = tf.keras.Model(inp, layers.Dense(3, "softmax")(x))
    model.compile(tf.keras.optimizers.Adam(1e-3), "categorical_crossentropy")
    target = np.eye(3)[y]
    rng = np.random.default_rng(seed)
    best, best_weights, wait = np.inf, None, 0
    for _ in range(200):
        view = rng.integers(1, acts.shape[0], len(train))
        model.fit(acts[view, train].astype("float32"), target[train], batch_size=32, epochs=1, verbose=0)
        loss = model.evaluate(acts[0, val].astype("float32"), target[val], verbose=0)
        if loss < best:
            best, best_weights, wait = loss, model.get_weights(), 0
        else:
            wait += 1
            if wait >= 20:
                break
    model.set_weights(best_weights)
    return model


def classifiers():
    return [DecisionTreeClassifier(random_state=0),
            KNeighborsClassifier(5),
            RandomForestClassifier(100, random_state=0, n_jobs=-1),
            HistGradientBoostingClassifier(max_iter=100, max_depth=3, max_bins=64,
                                           early_stopping=False, random_state=0)]


def ensemble_predict(X_train, y_train, X_test):
    prob = np.stack([c.fit(X_train, y_train).predict_proba(X_test) for c in classifiers()])
    return np.tensordot(np.asarray(WEIGHTS) / sum(WEIGHTS), prob, 1).argmax(-1)

## Run (resumable: finished folds are skipped)

In [ ]:
builder, preprocess = BACKBONES[BACKBONE]
net = tf.keras.Sequential([builder(include_top=False, weights="imagenet", input_shape=(SIZE, SIZE, 3)),
                           tf.keras.layers.GlobalAveragePooling2D()])
net.trainable = False

for condition in CONDITIONS:
    acts = None
    for k, (tr, te) in enumerate(folds):
        path = CHECK_DIR / f"{condition}_fold{k}.npz"
        if path.exists():
            continue
        if acts is None or condition in ("predicted", "shuffled"):  # raw and oracle inputs do not change across folds
            acts = activations(condition_images(condition, k), net, preprocess)
        order = np.random.default_rng(SEED + k).permutation(tr)
        n_val = len(tr) // 5
        head = train_head(acts, order[n_val:], order[:n_val], SEED + k)
        X = tf.keras.Model(head.input, head.get_layer("f512").output).predict(acts[0].astype("float32"), verbose=0)
        np.savez_compressed(path, cnn=head.predict(acts[0, te].astype("float32"), verbose=0).argmax(1),
                            ens=ensemble_predict(X[tr], y[tr], X[te]))
    print(condition, "done")

## Pooled results and McNemar tests

In [ ]:
def fold_predictions(backbone, condition, k, model):
    tr, te = folds[k]
    if backbone == "MobileNetV1":
        f = np.load(OUT_DIR / "features" / f"{condition}_fold{k}.npz")
        return f["cnn"] if model == "cnn" else ensemble_predict(f["X"][tr], y[tr], f["X"][te])
    return np.load(OUT_DIR / "backbone_check" / backbone / f"{condition}_fold{k}.npz")[model]


def pooled(backbone, condition, model):
    pred = np.full(len(y), -1)
    for k, (_, te) in enumerate(folds):
        pred[te] = fold_predictions(backbone, condition, k, model)
    return pred


def boot_ci(p, n=1000):
    rng = np.random.default_rng(0)
    f = [f1_score(y[i], p[i], average="macro") for i in (rng.integers(0, len(y), len(y)) for _ in range(n))]
    return np.percentile(f, 2.5), np.percentile(f, 97.5)


def mcnemar(a, b):
    ca, cb = a == y, b == y
    n10, n01 = int((ca & ~cb).sum()), int((~ca & cb).sum())
    return n10, n01, (binomtest(n10, n10 + n01).pvalue if n10 + n01 else 1.0)


rows, tests = [], []
for backbone in ["MobileNetV1", BACKBONE]:
    for model in ["cnn", "ens"]:
        preds = {c: pooled(backbone, c, model) for c in CONDITIONS}
        for condition, p in preds.items():
            lo, hi = boot_ci(p)
            rec = recall_score(y, p, average=None)
            rows.append(dict(dataset=DATASET, backbone=backbone, model=model, condition=condition,
                             macro_f1=f1_score(y, p, average="macro"), ci_low=lo, ci_high=hi,
                             recall_benign=rec[0], recall_malignant=rec[1], recall_normal=rec[2]))
        for a, b in [("oracle", "shuffled"), ("oracle", "raw"), ("predicted", "raw")]:
            n10, n01, p = mcnemar(preds[a], preds[b])
            tests.append(dict(dataset=DATASET, backbone=backbone, model=model, comparison=f"{a} vs {b}",
                              a_only=n10, b_only=n01, p_exact=p))

results = pd.DataFrame(rows).round(3)
tests = pd.DataFrame(tests).round(4)
results.to_csv(OUT_DIR / "backbone_results.csv", index=False)
tests.to_csv(OUT_DIR / "backbone_mcnemar.csv", index=False)
results.pivot_table(index=["model", "backbone"], columns="condition", values="macro_f1")[CONDITIONS]

In [ ]:
tests.pivot_table(index=["model", "backbone"], columns="comparison", values="p_exact")